# Brisk-Light Black Hole Imaging from GRMHD Simulations

Movie generation with the brisk-light prescription, parameterized by the
retained probability mass `p`:

- `p = 0`  : every step of band n is evaluated at the modal time t_n
- `0<p<1`  : step times inside the modal HDI are kept, tails clipped to the edge
- `p = 1`  : clipping bypassed, reduces to slow light

Geodesics are traced once and reused for every frame and every p.

In [1]:
using Jipole
using StaticArrays
using Printf

const MBH = 6.2e9

6.2e9

In [2]:
#const all_dumps_path = "/home/pedro/kharma_dumps/tmp.%05d.h5"
const all_dumps_path = "../../data/kharma_dumps/tmp.%05d.h5"

"../../data/kharma_dumps/tmp.%05d.h5"

In [3]:
const dump_first = 0
const dump_last  = 10
const ImageCadence = 10

dump_list  = [Printf.format(Printf.Format(all_dumps_path), i) 
              for i in dump_first:dump_last]
dump_times = [Jipole.Brisklight.get_dump_time(i, all_dumps_path) for i in dump_first:dump_last]

println("Total dumps available: ", length(dump_list))
println("Time range: $(dump_times[1]) M  →  $(dump_times[end]) M")

Total dumps available: 11
Time range: 1100.0026804973966 M  →  2100.0055612048927 M


In [4]:
trat_large       = 20.
const trat_small     = 1.
const beta_crit      = 1.0
const th_beg         = 1.74e-2
const sigma_cut      = 1.0
const sigma_cut_high = -1.0

-1.0

In [5]:
println("Initializing grid from: ", dump_list[1])
model = Jipole.Iharm.read_header(
    dump_list[1], MBH;
    brisk_light=true,
    slow_light=false,
    trat_small=trat_small,
    beta_crit=beta_crit, 
    th_beg=th_beg,
    sigma_cut=sigma_cut, 
    sigma_cut_high=sigma_cut_high
)
println("Model initialized successfully")

Initializing grid from: ../../data/kharma_dumps/tmp.00000.h5
Initializing grid from: ../../data/kharma_dumps/tmp.00000.h5


custom electron model loaded from dump file...
Using Funky Modified Kerr-Schild coordinates FMKS
MKS parameters a: 0.937500 hslope: 0.300000 Rin: 1.001876 Rout: 1000.000000
FMKS parameters poly_xt: 0.820000 poly_alpha: 14.000000 mks_smooth: 0.500000 poly_norm: 0.757817


Model initialized successfully


Grid start (startx): 1.874000951149755e-03, 0.000000000000000e+00, 0.000000000000000e+00 stop (stopx): 6.907755278982137e+00, 1.000000000000000e+00, 6.283185307179586e+00
grid dx: 5.395219748461709e-02, 7.812500000000000e-03, 6.283185307179586e+00


In [6]:
const ro      = 1000.0
const th      = 163.0
const phi     = 0.0

const res     = 128
const pixels_x = 128
const pixels_y = 128

const SourceD = 16.9e6 * Jipole.Constants.PC
const Rh      = 1 + sqrt(1. - model.a * model.a)
const freq    = 230e9

const DXsize  = SourceD / model.L_unit / Jipole.Constants.MUAS_PER_RAD * 160
const DYsize  = SourceD / model.L_unit / Jipole.Constants.MUAS_PER_RAD * 160
const fovx    = DXsize / ro
const fovy    = DYsize / ro

println("Resolution:        $pixels_x × $pixels_y pixels")
println("Inclination:       th = $(th)° (equivalent to $(90-th)° from jet axis)")
println("Spin:              a = $(model.a)")
println("Frequency:         230.0 GHz")
println("Field of view:     $(DXsize / model.L_unit / Jipole.Constants.MUAS_PER_RAD) μas")

Resolution:        128 × 128 pixels
Inclination:       th = 163.0° (equivalent to -73.0° from jet axis)
Spin:              a = 0.9375
Frequency:         230.0 GHz
Field of view:     2.338501443495858e-25 μas


In [7]:
using ProgressMeter

# Calculate the camera position in native coordinates
Xcamera = MVector{4,Float64}(Jipole.Camera.camera_position(ro, th, phi, model.a, model))

# Unitless frequency
const freq_unitless = freq * Jipole.Constants.HPL / (Jipole.Constants.ME * Jipole.Constants.CL * Jipole.Constants.CL)

# Initialize arrays for geodesics
nsteps = zeros(Int, pixels_x, pixels_y)
midplane_crossings = zeros(Int, pixels_x, pixels_y)

println("Camera setup complete")
println("  Position: ro=$ro, th=$th, phi=$phi")
println("  Frequency (unitless): $freq_unitless")

Camera setup complete
  Position: ro=1000.0, th=163.0, phi=0.0
  Frequency (unitless): 1.8614589389997047e-9


In [8]:
# Number of threads used in the calculation
println("Allocating workspaces for $pixels_x row-tasks...")
const maxnstep = 25000
dummy_svec = @SVector zeros(4)
dummy_traj = Jipole.GeoTypes.OfTrajS(0.0, dummy_svec, dummy_svec, dummy_svec, dummy_svec)

task_trajs = [Vector{Jipole.GeoTypes.OfTrajS}(undef, maxnstep) for _ in 1:pixels_x]
for i in 1:pixels_x
    for k in 1:maxnstep
        task_trajs[i][k] = dummy_traj
    end
end

# This will hold the exact number of steps for each pixel.
const all_geodesics = Matrix{Vector{Jipole.GeoTypes.OfTrajS}}(undef, pixels_x, pixels_y)

println("Tracing Geodesics...")
p = Progress(pixels_x * pixels_y; desc="Raytracing Image...", showspeed=true, barlen=30)

Threads.@threads for i in 0:(pixels_x - 1)
    my_traj = task_trajs[i + 1]
    
    for j in 0:(pixels_y - 1)
        nstep, midplane_crossings[i+1,j+1] = Jipole.Geodesics.get_pixel(
            my_traj, i, j, Xcamera, 
            fovx, fovy, freq_unitless, 
            pixels_x, pixels_y, model.a, 
            Rh, model.rmax_geo, model
        ) 
        nsteps[i+1, j+1] = nstep
        
        # Save to permanent storage
        all_geodesics[i + 1, j + 1] = my_traj[1:nstep]
        
        ProgressMeter.next!(p)
    end
end
finish!(p)

println("Geodesics computed successfully")

Allocating workspaces for 128 row-tasks...
Tracing Geodesics...


Raytracing Image... 100%|██████████████████████████████| Time: 0:00:04 ( 0.26 ms/it)


Geodesics computed successfully


In [9]:
# Brisk-light run state.
# OfBriskLight now carries `hdi_intervals` (T_{n,p} per band) and
# `band_time_ranges` (min/max sampled step time per band, needed because the
# p = 1 HDI is infinite).

p_test = 1.0   # 0.0, 0.25, 0.5, 0.75, 1.0

n_bands = 3

params_brisklight = Jipole.Brisklight.OfBriskLight(
    n_bands,
    zeros(Float64, n_bands + 1),        # modal_times
    Vector{Tuple}(undef, n_bands + 1),  # hdi_intervals
    Vector{Tuple}(undef, n_bands + 1),  # band_time_ranges
    p_test,                             # p
    ImageCadence,
    0.0                                 # t_obs
)

println("Brisk-light parameters initialized with p = $p_test")

Brisk-light parameters initialized with p = 1.0


In [ ]:
# Render. Output goes to ../../data/Images/BriskLight_p$(p), so different p
# values never overwrite each other.

tA = dump_times[1]
tB = dump_times[end]
tf = dump_times[end]

println("\nRendering brisk-light with p = $(params_brisklight.p)")
println("  Dump range     : [$tA, $tB] M")
println("  Output         : ../../data/Images/BriskLight_p$(params_brisklight.p)")

Jipole.Brisklight.process_brisklight_images!(
    params_brisklight,
    Vector{Jipole.Iharm.IharmData}(undef, 1),   # simulation_data (unused)
    all_geodesics,
    nsteps,
    midplane_crossings,
    model,
    pixels_x,
    pixels_y,
    freq,
    trat_large,
    all_dumps_path,
    dump_list,
    dump_times,
    tA, tB, tf;
    pixel_stride = 4
)

println("\nRendering complete!")


Rendering brisk-light with p = 1.0
  Dump range     : [1100.0026804973966, 2100.0055612048927] M
  Output         : ../../data/Images/BriskLight_p1.0


[ Info: Brisk-light: band 0 -> p=1.0 -> t_modal_0 = -1008.2993781218604 M, HDI = [-Inf, Inf] M (485396 step samples)
[ Info: Brisk-light: band 1 -> p=1.0 -> t_modal_1 = -1035.0036798105605 M, HDI = [-Inf, Inf] M (622142 step samples)
[ Info: Brisk-light: band 2 -> p=1.0 -> t_modal_2 = -1068.6071811273027 M, HDI = [-Inf, Inf] M (17386 step samples)


Loading data from '../../data/kharma_dumps/tmp.00000.h5' into 'Iharm' module...


[ Info: Brisk-light: band 3 -> p=1.0 -> t_modal_3 = -1088.8318585804102 M, HDI = [-Inf, Inf] M (4807 step samples)
[ Info: Brisk-light: processing frame at t_obs = 2188.8345390778068 M (p = 1.0)


All primitives successfully loaded. Dimensions: (128, 128, 1)
Loading data from '../../data/kharma_dumps/tmp.00001.h5' into 'Iharm' module...
All primitives successfully loaded. Dimensions: (128, 128, 1)
Loading data from '../../data/kharma_dumps/tmp.00002.h5' into 'Iharm' module...
All primitives successfully loaded. Dimensions: (128, 128, 1)


...


[ Info: Brisk-light: processing frame at t_obs = 2898.8345390778068 M (p = 1.0)
[ Info:   Band window updated: t_target = 1807.0123934306391 M -> dumps[(7, 8, 9)] at t = [1700.0013761363548, 1900.001386364225] M
Brisk-light t_obs = 2898.8345390778068 M 100%|██████████████████████████████| Time: 0:00:48 ( 2.97 ms/it)


Brisk-light: image saved -> ../../data/Images/BriskLight_p1.0\BriskImage.02899.txt
Loading data from '../../data/kharma_dumps/tmp.00009.h5' into 'Iharm' module...


[ Info: Brisk-light: processing frame at t_obs = 2908.8345390778068 M (p = 1.0)


All primitives successfully loaded. Dimensions: (128, 128, 1)


[ Info:   Band window updated: t_target = 1904.8253185935916 M -> dumps[(8, 9, 10)] at t = [1800.0015230265558, 2000.007157193934] M
[ Info:   Band window updated: t_target = 1808.459598048559 M -> dumps[(7, 8, 9)] at t = [1700.0013761363548, 1900.001386364225] M
Brisk-light t_obs = 2908.8345390778068 M  27%|█████████                     |  ETA: 0:00:33 ( 2.78 ms/it)

## Diagnostics

Run these before trusting any p sweep. Each has a binary pass criterion.

In [ ]:
# --- Diagnostic 1: retained width W_n(p) -------------------------------------
# PASS: W_n(0) == 0 exactly, W_n monotonically increasing, t_modal constant in p.
# If W_n(1) is much smaller than the dump spacing, p cannot be resolved with
# this dataset regardless of the code being correct.

using Printf

dT = dump_times[2] - dump_times[1]
println("dump spacing dT = $dT M\n")

for p in [0.0, 0.25, 0.5, 0.75, 0.99, 1.0]
    Jipole.Brisklight.compute_band_modal_times!(
        midplane_crossings, all_geodesics, nsteps,
        pixels_x, pixels_y, params_brisklight, model, p; pixel_stride = 4)
    for n in 0:params_brisklight.n_bands
        tL, tR = params_brisklight.hdi_intervals[n+1]
        rL, rR = params_brisklight.band_time_ranges[n+1]
        W = min(tR, rR) - max(tL, rL)
        @printf("p=%.2f  band %d   t_modal=%9.3f   W=%8.3f M   (W/dT = %.2f)\n",
                p, n, params_brisklight.modal_times[n+1], W, W/dT)
    end
    println()
end

In [ ]:
# --- Diagnostic 2: fraction of steps clipped ---------------------------------
# PASS: with p = 1 the clipped fraction must be 0.00 %. Anything else means the
# clipping map is not the identity and p = 1 will not converge to slow light.

using Statistics

Jipole.Brisklight.compute_band_modal_times!(
    midplane_crossings, all_geodesics, nsteps,
    pixels_x, pixels_y, params_brisklight, model, 1.0; pixel_stride = 4)

Rh_check = 1.0 + sqrt(1.0 - model.a * model.a)
band_ts = Jipole.Brisklight.collect_band_step_times(
    midplane_crossings, all_geodesics, nsteps,
    pixels_x, pixels_y, params_brisklight.n_bands, model, Rh_check; pixel_stride = 4)

for n in 0:params_brisklight.n_bands
    ts = band_ts[n+1]
    length(ts) < 2 && continue
    tL, tR = params_brisklight.hdi_intervals[n+1]
    nclip = count(t -> clamp(t, tL, tR) != t, ts)
    @printf("band %d: HDI = [%.3f, %.3f]  step range = [%.3f, %.3f]  clipped = %.2f %%\n",
            n, tL, tR, minimum(ts), maximum(ts), 100*nclip/length(ts))
end

In [ ]:
# --- Diagnostic 3: is time interpolation actually enabled? -------------------
# PASS: j must VARY across the probed times.
# If j is constant, `interp_scalar_time` is still gated on model.slow_light:
# the iharm.jl edits were not applied, or Jipole was not recompiled.
#
# THREE snapshots, not two: set_tinterp_ns (iharm.jl:724) indexes data[3]
# whenever X[1] >= data[2].t, so a 2-element vector raises a BoundsError.

using StaticArrays, Printf

d1 = Jipole.Iharm.load_data(dump_list[2], trat_large, model)
d2 = Jipole.Iharm.load_data(dump_list[3], trat_large, model)
d3 = Jipole.Iharm.load_data(dump_list[4], trat_large, model)
data_vec = [d1, d2, d3]
println("snapshot times: ", (d1.t, d2.t, d3.t))

# pick the brightest emitting step, not an arbitrary corner pixel
best = (0.0, 0, 0, 0)
for i in 1:8:pixels_x, j in 1:8:pixels_y
    tr = all_geodesics[i,j]
    for k in 1:nsteps[i,j]
        Jipole.Radiation.radiating_region(tr[k].X, model, Rh) || continue
        X = MVector{4,Float64}(tr[k].X); X[1] = d2.t   # inside the valid domain
        jj, _ = Jipole.Radiation.get_jk(X, tr[k].Kcon, freq, model.a, model, data_vec)
        jj > best[1] && (best = (jj, i, j, k))
    end
end
_, bi, bj, bk = best
best[1] == 0.0 && error("no emitting step found: check sigma_cut / th_beg / data loading")
println("probing pixel ($bi,$bj) step $bk, geometric X[1] = $(all_geodesics[bi,bj][bk].X[1])")

tr = all_geodesics[bi,bj]
for f in 0.0:0.25:1.0
    t = d1.t + f * (d3.t - d1.t)
    X = MVector{4,Float64}(tr[bk].X); X[1] = t
    jj, kk = Jipole.Radiation.get_jk(X, tr[bk].Kcon, freq, model.a, model, data_vec)
    @printf("t = %10.3f   j = %.10e   k = %.10e\n", t, jj, kk)
end

In [ ]:
# --- p sweep ----------------------------------------------------------------
# Only run this after Diagnostics 1-3 pass. Each p writes to its own directory.

for p_value in [0.0, 0.5, 1.0]
    println("\n" * "="^60)
    println("Rendering with p = $p_value")
    println("="^60)

    params_brisklight.p = p_value

    Jipole.Brisklight.process_brisklight_images!(
        params_brisklight,
        Vector{Jipole.Iharm.IharmData}(undef, 1),
        all_geodesics, nsteps, midplane_crossings, model,
        pixels_x, pixels_y, freq, trat_large,
        all_dumps_path, dump_list, dump_times,
        tA, tB, tf; pixel_stride = 4
    )
end

## Visualization of Output Images

In [10]:
using DelimitedFiles
using CairoMakie
using Printf

brisk_dir  = "../../data/Images/BriskLight_p1.0"
ref_dir    = "../../data/Images/SlowLight"
gif_out    = "C:\\Users\\danyp\\OneDrive\\Escritorio\\CosasPater\\jipoleProyect\\data\\Images\\comparison_p1.gif"

brisk_files = sort(filter(f -> startswith(basename(f), "BriskImage.") && endswith(f, ".txt"),
                           readdir(brisk_dir, join=true)))
ref_files  = sort(filter(f -> startswith(basename(f), "Image.") && endswith(f, ".txt"),
                           readdir(ref_dir, join=true)))

println("Found $(length(brisk_files)) Brisk files and $(length(ref_files)) Slow files.")

# --- Extraer el índice numérico del nombre de archivo ---
function extract_index(path)
    m = match(r"(\d+)(?=\.txt$)", basename(path))
    return parse(Int, m.match)
end

brisk_indices = extract_index.(brisk_files)
ref_indices  = extract_index.(ref_files)

# --- Alinear el inicio de brisk al índice más cercano al primer índice de slow ---
ref_start = ref_indices[1]
closest_pos = argmin(abs.(brisk_indices .- ref_start))
println("Reference (p=1) empieza en índice $(ref_start). Brisk light se alinea desde índice $(brisk_indices[closest_pos]) (posición $(closest_pos) en la lista).")

brisk_files   = brisk_files[closest_pos:end]
brisk_indices = brisk_indices[closest_pos:end]

if length(brisk_files) != length(ref_files)
    println("The script will pair them up until it runs out of files in the shorter list.")
end

n_pairs = min(length(brisk_files), length(ref_files))
eps_val = 1e-30
vmin, vmax = 1e-12, 1e-5

fig = Figure(size=(1800, 500))

record(fig, gif_out, 1:n_pairs; framerate=10) do i
    empty!(fig)

    brisk_path = brisk_files[i]
    ref_path  = ref_files[i]
    brisk_filename = basename(brisk_path)
    ref_filename  = basename(ref_path)

    I_ref  = readdlm(ref_path)
    I_brisk = readdlm(brisk_path)
    if size(I_brisk) != size(I_ref)
        I_brisk = reshape(I_brisk, size(I_ref))
    end

    rel_err = abs.(I_brisk .- I_ref) ./ (abs.(I_ref) .+ eps_val)
    nmse = sum((I_ref .- I_brisk).^2) / sum(I_ref.^2)

    ax1 = Axis(fig[1, 1], title = "Slow Light | $(ref_filename)")
    hm1 = heatmap!(ax1, I_ref, colormap=:afmhot, colorscale=log10, colorrange=(vmin, vmax))
    Colorbar(fig[1, 2], hm1)

    ax2 = Axis(fig[1, 3], title = "Brisk Light p = 1| $(brisk_filename)")
    hm2 = heatmap!(ax2, I_brisk, colormap=:afmhot, colorscale=log10, colorrange=(vmin, vmax))
    Colorbar(fig[1, 4], hm2)

    ax3 = Axis(fig[1, 5], title = "Relative Error")
    hm3 = heatmap!(ax3, rel_err, colormap=:viridis, colorscale=log10, colorrange=(1e-6, 10.0))
    Colorbar(fig[1, 6], hm3)

    text!(ax3, 0.05, 0.95;
        text = @sprintf("NMSE = %.2e", nmse),
        space = :relative,
        color = :white,
        fontsize = 16,
        font = :bold,
        align = (:left, :top))

    println("[Pair $(lpad(i-1,4,'0'))] Frame added for $(brisk_filename) vs $(ref_filename)")
end

println("Done! GIF saved to $(gif_out)")

Found 72 Brisk files and 64 Slow files.
Reference (p=1) empieza en índice 2271. Brisk light se alinea desde índice 2269 (posición 9 en la lista).
[Pair 0000] Frame added for BriskImage.02269.txt vs Image.02271.txt
[Pair 0001] Frame added for BriskImage.02279.txt vs Image.02281.txt
[Pair 0002] Frame added for BriskImage.02289.txt vs Image.02291.txt
[Pair 0003] Frame added for BriskImage.02299.txt vs Image.02301.txt
[Pair 0004] Frame added for BriskImage.02309.txt vs Image.02311.txt
[Pair 0005] Frame added for BriskImage.02319.txt vs Image.02321.txt
[Pair 0006] Frame added for BriskImage.02329.txt vs Image.02331.txt
[Pair 0007] Frame added for BriskImage.02339.txt vs Image.02341.txt
[Pair 0008] Frame added for BriskImage.02349.txt vs Image.02351.txt
[Pair 0009] Frame added for BriskImage.02359.txt vs Image.02361.txt
[Pair 0010] Frame added for BriskImage.02369.txt vs Image.02371.txt
[Pair 0011] Frame added for BriskImage.02379.txt vs Image.02381.txt
[Pair 0012] Frame added for BriskImage

In [11]:
using DelimitedFiles
using CairoMakie
using Printf

brisk_dir  = "../../data/Images/BriskLight_p0.0"
ref_dir    = "../../data/Images/BriskLight_p1.0"
gif_out    = "C:\\Users\\danyp\\OneDrive\\Escritorio\\CosasPater\\jipoleProyect\\data\\Images\\comparison_p0p1.gif"

brisk_files = sort(filter(f -> startswith(basename(f), "BriskImage.") && endswith(f, ".txt"),
                           readdir(brisk_dir, join=true)))
ref_files  = sort(filter(f -> startswith(basename(f), "BriskImage.") && endswith(f, ".txt"),
                           readdir(ref_dir, join=true)))

println("Found $(length(brisk_files)) Brisk files and $(length(ref_files)) Slow files.")

# --- Extraer el índice numérico del nombre de archivo ---
function extract_index(path)
    m = match(r"(\d+)(?=\.txt$)", basename(path))
    return parse(Int, m.match)
end

brisk_indices = extract_index.(brisk_files)
ref_indices  = extract_index.(ref_files)

# --- Alinear el inicio de brisk al índice más cercano al primer índice de slow ---
ref_start = ref_indices[1]
closest_pos = argmin(abs.(brisk_indices .- ref_start))
println("Reference (p=1) empieza en índice $(ref_start). Brisk light se alinea desde índice $(brisk_indices[closest_pos]) (posición $(closest_pos) en la lista).")

brisk_files   = brisk_files[closest_pos:end]
brisk_indices = brisk_indices[closest_pos:end]

if length(brisk_files) != length(ref_files)
    println("The script will pair them up until it runs out of files in the shorter list.")
end

n_pairs = min(length(brisk_files), length(ref_files))
eps_val = 1e-30
vmin, vmax = 1e-12, 1e-5

fig = Figure(size=(1800, 500))

record(fig, gif_out, 1:n_pairs; framerate=10) do i
    empty!(fig)

    brisk_path = brisk_files[i]
    ref_path  = ref_files[i]
    brisk_filename = basename(brisk_path)
    ref_filename  = basename(ref_path)

    I_ref  = readdlm(ref_path)
    I_brisk = readdlm(brisk_path)
    if size(I_brisk) != size(I_ref)
        I_brisk = reshape(I_brisk, size(I_ref))
    end

    rel_err = abs.(I_brisk .- I_ref) ./ (abs.(I_ref) .+ eps_val)
    nmse = sum((I_ref .- I_brisk).^2) / sum(I_ref.^2)

    ax1 = Axis(fig[1, 1], title = "Brisk Light p = 1 | $(ref_filename)")
    hm1 = heatmap!(ax1, I_ref, colormap=:afmhot, colorscale=log10, colorrange=(vmin, vmax))
    Colorbar(fig[1, 2], hm1)

    ax2 = Axis(fig[1, 3], title = "Brisk Light p = 0| $(brisk_filename)")
    hm2 = heatmap!(ax2, I_brisk, colormap=:afmhot, colorscale=log10, colorrange=(vmin, vmax))
    Colorbar(fig[1, 4], hm2)

    ax3 = Axis(fig[1, 5], title = "Relative Error")
    hm3 = heatmap!(ax3, rel_err, colormap=:viridis, colorscale=log10, colorrange=(1e-6, 10.0))
    Colorbar(fig[1, 6], hm3)

    text!(ax3, 0.05, 0.95;
        text = @sprintf("NMSE = %.2e", nmse),
        space = :relative,
        color = :white,
        fontsize = 16,
        font = :bold,
        align = (:left, :top))

    println("[Pair $(lpad(i-1,4,'0'))] Frame added for $(brisk_filename) vs $(ref_filename)")
end

println("Done! GIF saved to $(gif_out)")

Found 78 Brisk files and 72 Slow files.
Reference (p=1) empieza en índice 2189. Brisk light se alinea desde índice 2189 (posición 1 en la lista).
The script will pair them up until it runs out of files in the shorter list.
[Pair 0000] Frame added for BriskImage.02189.txt vs BriskImage.02189.txt
[Pair 0001] Frame added for BriskImage.02199.txt vs BriskImage.02199.txt
[Pair 0002] Frame added for BriskImage.02209.txt vs BriskImage.02209.txt
[Pair 0003] Frame added for BriskImage.02219.txt vs BriskImage.02219.txt
[Pair 0004] Frame added for BriskImage.02229.txt vs BriskImage.02229.txt
[Pair 0005] Frame added for BriskImage.02239.txt vs BriskImage.02239.txt
[Pair 0006] Frame added for BriskImage.02249.txt vs BriskImage.02249.txt
[Pair 0007] Frame added for BriskImage.02259.txt vs BriskImage.02259.txt
[Pair 0008] Frame added for BriskImage.02269.txt vs BriskImage.02269.txt
[Pair 0009] Frame added for BriskImage.02279.txt vs BriskImage.02279.txt
[Pair 0010] Frame added for BriskImage.02289.tx